# One-sided paired statistical testing for `outputs/opt_prompt_old`

This notebook follows the scoring and bootstrap workflow from `statistical_testing.ipynb`, but is set up for selecting one main output file and comparing it against five or more files from `/storage2/home/aunabilchakma/codes/RE_Prompt_optimizer/outputs/opt_prompt_old`.

The default one-sided alternative is `main F1 > comparison F1`.


In [1]:
from pathlib import Path
import ast
import os
import sys
import numpy as np

sys.path.insert(0, '/storage2/home/aunabilchakma/codes/relation_extraction')
import scorer

SOURCE_DIR = Path('/storage2/home/aunabilchakma/codes/RE_Prompt_optimizer/outputs/opt_prompt_old')
assert SOURCE_DIR.exists(), f'Missing directory: {SOURCE_DIR}'


In [4]:
def evaluate_in_chunks(golds, preds, scorer, chunk_size=30000):
    n_chunks = (len(golds) // chunk_size) + 1
    precisions, recalls, f1s = [], [], []

    for i in range(n_chunks):
        start = i * chunk_size
        end = (i + 1) * chunk_size if i < n_chunks - 1 else len(golds)

        g_chunk = golds[start:end]
        p_chunk = preds[start:end]

        if len(g_chunk) == 0:
            continue

        sc = scorer.score(g_chunk, p_chunk, False)
        precisions.append(sc[0])
        recalls.append(sc[1])
        f1s.append(sc[2])

    precisions = np.array(precisions)
    recalls = np.array(recalls)
    f1s = np.array(f1s)

    return {
        'mean_P': np.mean(precisions) * 100,
        'std_P': np.std(precisions) * 100,
        'mean_R': np.mean(recalls) * 100,
        'std_R': np.std(recalls) * 100,
        'mean_F1': np.mean(f1s) * 100,
        'std_F1': np.std(f1s) * 100,
    }


In [5]:
def _collect_after_marker(source, marker):
    values = []
    ready = False
    with open(source, 'r') as f:
        for line in f:
            line = line.strip()
            if ready:
                values.append(line)
            ready = marker in line if marker.startswith('@') else line == marker
    return values


def get_scores(source, do_print=False):
    source = Path(source)
    p_list = _collect_after_marker(source, '@@@@@@@@@@')
    q_list = _collect_after_marker(source, '$$$$$ query relation')
    rl_list = _collect_after_marker(source, '$$$$$ relation list')
    r_list = _collect_after_marker(source, '$$$$$ votes')

    if not (len(p_list) == len(q_list) == len(rl_list) == len(r_list)):
        raise ValueError(
            f'Parsed lengths differ for {source.name}: '
            f'prompts={len(p_list)}, queries={len(q_list)}, relation_lists={len(rl_list)}, votes={len(r_list)}'
        )

    p_list = [ast.literal_eval(x) for x in p_list]
    rl_list = [ast.literal_eval(x) for x in rl_list]
    r_votes = [ast.literal_eval(x) for x in r_list]

    preds = []
    for votes, relations in zip(r_votes, rl_list):
        dec = None
        dec_list = []
        multiple = False

        for vote, relation in zip(votes, relations):
            if vote == 'yes':
                if dec is not None:
                    multiple = True
                dec = relation
                dec_list.append(relation)

        if dec is None:
            dec = 'no_relation'
        elif multiple:
            dec = '#multiple#'

        preds.append(dec)

    golds = []
    for query_relation, relation_list in zip(q_list, rl_list):
        if query_relation in relation_list:
            assert query_relation != 'no_relation'
            golds.append(query_relation)
        else:
            golds.append('no_relation')

    res = evaluate_in_chunks(golds, preds, scorer, chunk_size=30000)

    if do_print:
        print(source.name)
        print(len(golds))
        print(
            f"{res['mean_P']:04.1f} +/- {res['std_P']:4.2f}	"
            f"{res['mean_R']:04.1f} +/- {res['std_R']:4.2f}	"
            f"{res['mean_F1']:04.1f} +/- {res['std_F1']:4.2f}"
        )
        print(
            f"& {res['mean_P']:04.1f} & {res['std_P']:4.2f} & "
            f"{res['mean_R']:04.1f} & {res['std_R']:4.2f} & "
            f"{res['mean_F1']:04.1f} & {res['std_F1']:4.2f}"
        )

    return res, preds, golds


In [6]:
def bootstrap_f1_pvalue(
    p_main,
    p_other,
    golds,
    scorer,
    K=1000,
    chunk_size=150000,
    seed=42,
    alternative='greater',
):
    '''
    One-sided paired bootstrap test on F1.

    alternative='greater': H1 is main F1 > other F1; p = Pr(diff <= 0)
    alternative='less':    H1 is main F1 < other F1; p = Pr(diff >= 0)
    alternative='two-sided' is included for convenience.
    '''
    rng = np.random.default_rng(seed)
    n = len(golds)

    res_main_full = evaluate_in_chunks(golds, p_main, scorer, chunk_size)
    res_other_full = evaluate_in_chunks(golds, p_other, scorer, chunk_size)
    obs_diff = res_main_full['mean_F1'] - res_other_full['mean_F1']

    f1_diffs = np.empty(K)
    for k in range(K):
        idx = rng.integers(0, n, size=n)

        golds_bs = [golds[i] for i in idx]
        preds_main_bs = [p_main[i] for i in idx]
        preds_other_bs = [p_other[i] for i in idx]

        res_main = evaluate_in_chunks(golds_bs, preds_main_bs, scorer, chunk_size)
        res_other = evaluate_in_chunks(golds_bs, preds_other_bs, scorer, chunk_size)
        f1_diffs[k] = res_main['mean_F1'] - res_other['mean_F1']

    if alternative == 'greater':
        p_value = np.mean(f1_diffs <= 0)
    elif alternative == 'less':
        p_value = np.mean(f1_diffs >= 0)
    elif alternative == 'two-sided':
        p_value = 2 * min(np.mean(f1_diffs <= 0), np.mean(f1_diffs >= 0))
        p_value = min(float(p_value), 1.0)
    else:
        raise ValueError("alternative must be one of: 'greater', 'less', 'two-sided'")

    return {
        'p_value': float(p_value),
        'alternative': alternative,
        'observed_diff': float(obs_diff),
        'main_F1': float(res_main_full['mean_F1']),
        'other_F1': float(res_other_full['mean_F1']),
        'bootstrap_mean_diff': float(f1_diffs.mean()),
        'ci_90': np.percentile(f1_diffs, [5, 95]),
        'ci_95': np.percentile(f1_diffs, [2.5, 97.5]),
        'ci_99': np.percentile(f1_diffs, [0.5, 99.5]),
    }


## List files

Run this cell first to see indexes. Then either use the default `MAIN_FILE` / `COMPARE_FILES` below or edit them manually.


In [18]:
files = sorted([p.name for p in SOURCE_DIR.glob('*.txt') if "cross" not in p.name])
print(f'{len(files)} files in {SOURCE_DIR}')
for ix, name in enumerate(files):
    print(f'{ix:03d}: {name}')


90 files in /storage2/home/aunabilchakma/codes/RE_Prompt_optimizer/outputs/opt_prompt_old
000: fewrel_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
001: fewrel_gemma_etgpo_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
002: fewrel_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
003: fewrel_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
004: fewrel_gemma_evoprompt_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
005: fewrel_gemma_evoprompt_node_x_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.

In [ ]:
# 90 files in /storage2/home/aunabilchakma/codes/RE_Prompt_optimizer/outputs/opt_prompt_old
next19# 000: fewrel_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 001: fewrel_gemma_etgpo_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 002: fewrel_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 003: fewrel_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
next17# 004: fewrel_gemma_evoprompt_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 005: fewrel_gemma_evoprompt_node_x_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 006: fewrel_gemma_evoprompt_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 007: fewrel_gemma_evoprompt_node_x_greater_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 008: fewrel_gemma_evoprompt_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 009: fewrel_gemma_evoprompt_node_y_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
next18# 010: fewrel_gemma_evoprompt_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 011: fewrel_gemma_evoprompt_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 012: fewrel_gemma_evoprompt_node_y_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 013: fewrel_gemma_initial_prompt_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 014: fewrel_gemma_rpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 015: fewrel_gemma_rpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 016: fewrel_gemma_rpo_node_x_greater_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
next15# 017: fewrel_gemma_rpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 018: fewrel_gemma_rpo_node_y_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
next16# 019: fewrel_gemma_rpo_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 020: fewrel_gemma_rpo_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 021: fewrel_gemma_rpo_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 022: fewrel_qwen_etgpo_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 023: fewrel_qwen_etgpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 024: fewrel_qwen_etgpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
next14# 025: fewrel_qwen_etgpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 026: fewrel_qwen_evoprompt_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 027: fewrel_qwen_evoprompt_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 028: fewrel_qwen_evoprompt_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 029: fewrel_qwen_evoprompt_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
next13# 030: fewrel_qwen_evoprompt_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 031: fewrel_qwen_initial_prompt_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 032: fewrel_qwen_rpo_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
next12# 033: fewrel_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 034: fewrel_qwen_rpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 035: fewrel_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 036: fewrel_qwen_rpo_node_y_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
next11# 037: fewrel_qwen_rpo_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 038: fewrel_qwen_rpo_node_y_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 039: fewrel_qwen_rpo_node_y_greater-tg_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 040: tacred_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
next10# 041: tacred_gemma_etgpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 042: tacred_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
next8# 043: tacred_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 044: tacred_gemma_evoprompt_node_x_gradpo-prob_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 045: tacred_gemma_evoprompt_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 046: tacred_gemma_evoprompt_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 047: tacred_gemma_evoprompt_node_y_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 048: tacred_gemma_evoprompt_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 049: tacred_gemma_evoprompt_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
next9# 050: tacred_gemma_evoprompt_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 051: tacred_gemma_evoprompt_node_y_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 052: tacred_gemma_evoprompt_node_y_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 053: tacred_gemma_initial_prompt_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 054: tacred_gemma_rpo_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 055: tacred_gemma_rpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 056: tacred_gemma_rpo_node_x_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 057: tacred_gemma_rpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 058: tacred_gemma_rpo_node_y_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 059: tacred_gemma_rpo_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
next7# 060: tacred_gemma_rpo_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 061: tacred_gemma_rpo_node_y_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 062: tacred_qwen_etgpo_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 063: tacred_qwen_etgpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 064: tacred_qwen_etgpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 065: tacred_qwen_etgpo_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 066: tacred_qwen_etgpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 067: tacred_qwen_etgpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 068: tacred_qwen_evoprompt_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 069: tacred_qwen_evoprompt_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 070: tacred_qwen_evoprompt_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 071: tacred_qwen_evoprompt_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 072: tacred_qwen_evoprompt_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 073: tacred_qwen_evoprompt_node_y_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 074: tacred_qwen_evoprompt_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 075: tacred_qwen_evoprompt_node_y_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 076: tacred_qwen_evoprompt_node_y_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 077: tacred_qwen_initial_prompt_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 078: tacred_qwen_rpo_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 079: tacred_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 080: tacred_qwen_rpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 081: tacred_qwen_rpo_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 082: tacred_qwen_rpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 083: tacred_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 084: tacred_qwen_rpo_node_y_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 085: tacred_qwen_rpo_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 086: tacred_qwen_rpo_node_y_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 087: tacred_qwen_rpo_node_y_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 088: tacred_qwen_rpo_node_y_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 089: tacred_qwen_rpo_node_y_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

In [ ]:

next19# 000: 
next17# 004: 
next18# 010: 
next15# 017: 
next16# 019: 
next14# 025: 
next13# 030: 
next12# 033: 
next11# 037: 
next10# 041: 
next8# 043: 
next9# 050: 
next7# 060:

## Select one main file and five or more comparison files

Defaults below use the first sorted file as `MAIN_FILE` and the next five files as comparisons. For a stricter paired test, compare files from the same dataset/model split so the gold labels and example order match.


In [25]:
# MAIN_prefixes = ["tacred_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt",
#                "tacred_qwen_rpo_node_y_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt",
#                "tacred_qwen_evoprompt_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt",
#                "tacred_qwen_evoprompt_node_y_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt",
#                "tacred_qwen_etgpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt",
#                "tacred_gemma_rpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt",
#                "tacred_gemma_rpo_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt", 
#                "tacred_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt", 
#                "tacred_gemma_evoprompt_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt", 
#                "tacred_gemma_etgpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt", 
#                "fewrel_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt", 
#                "fewrel_qwen_rpo_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt", 
#                "fewrel_qwen_evoprompt_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt", 
#                "fewrel_qwen_etgpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt", 
#                "fewrel_gemma_rpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt", 
#                "fewrel_gemma_rpo_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt", 
#                "fewrel_gemma_evoprompt_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt", 
#                "fewrel_gemma_evoprompt_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt", 
#                "fewrel_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt", 
# ]

MAIN_prefixes = ["fewrel_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt"]

cnt = 1
for main_pref in MAIN_prefixes:
    print(f"Currently doing: {cnt}")
    cnt += 1
    MAIN_FILE = main_pref
    comp_pref = "_".join(main_pref.split("_")[:5])
    COMPARE_FILES = [fa for fa in files if (MAIN_FILE not in fa and comp_pref in fa)]

    ALTERNATIVE = 'greater'  # 'greater' means MAIN_FILE F1 > comparison F1
    K = 1000
    SEED = 42

    print('MAIN_FILE:')
    print(MAIN_FILE)
    print()
    print('COMPARE_FILES:')
    for name in COMPARE_FILES:
        print('-', name)

    main_score, main_preds, main_golds = get_scores(SOURCE_DIR / MAIN_FILE, do_print=True)

    results = {}
    for ix, compare_file in enumerate(COMPARE_FILES, start=1):
        if compare_file != "fewrel_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt":
            continue

        print()
        print('=' * 120)
        print(f'[{ix}/{len(COMPARE_FILES)}] comparing against: {compare_file}')
        other_score, other_preds, other_golds = get_scores(SOURCE_DIR / compare_file, do_print=False)

        if main_golds != other_golds:
            raise ValueError(
                'Gold labels differ between MAIN_FILE and comparison file. '
                'A paired test requires the same examples in the same order. '
                f'Failed comparison: {compare_file}'
            )

        res = bootstrap_f1_pvalue(
            main_preds,
            other_preds,
            main_golds,
            scorer=scorer,
            K=2000,
            seed=SEED,
            alternative=ALTERNATIVE,
        )
        results[compare_file] = res
        print(res)

Currently doing: 1
MAIN_FILE:
fewrel_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

COMPARE_FILES:
- fewrel_qwen_rpo_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
- fewrel_qwen_rpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
- fewrel_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
fewrel_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
42.9 +/- 1.27	30.7 +/- 0.64	35.8 +/- 0.80
& 42.9 & 1.27 & 30.7 & 0.64 & 35.8 & 0.80

[3/3] comparing against: fewrel_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-Fal

In [ ]:
fewrel_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

In [ ]:
# Currently doing: 1
# MAIN_FILE:
# tacred_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt

# COMPARE_FILES:
# - tacred_qwen_rpo_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# - tacred_qwen_rpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# - tacred_qwen_rpo_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# - tacred_qwen_rpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# - tacred_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# tacred_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# 150000
# 29.5 +/- 1.25	28.5 +/- 1.66	29.0 +/- 1.37
# & 29.5 & 1.25 & 28.5 & 1.66 & 29.0 & 1.37

# ========================================================================================================================
# [1/5] comparing against: tacred_qwen_rpo_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 2.4951744441968913, 'main_F1': 28.99657624440347, 'other_F1': 26.50140180020658, 'bootstrap_mean_diff': 2.5005501455609385, 'ci_90': array([1.83875353, 3.13002292]), 'ci_95': array([1.75514336, 3.24822116]), 'ci_99': array([1.52112919, 3.53576129])}

# ========================================================================================================================
# [2/5] comparing against: tacred_qwen_rpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.6358353997282897, 'main_F1': 28.99657624440347, 'other_F1': 27.36074084467518, 'bootstrap_mean_diff': 1.641240918254628, 'ci_90': array([1.11344684, 2.18191449]), 'ci_95': array([0.98138448, 2.32460189]), 'ci_99': array([0.66813182, 2.55175333])}

# ========================================================================================================================
# [3/5] comparing against: tacred_qwen_rpo_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.841002974840567, 'main_F1': 28.99657624440347, 'other_F1': 27.155573269562904, 'bootstrap_mean_diff': 1.8441320309426656, 'ci_90': array([1.32213347, 2.38652109]), 'ci_95': array([1.19940529, 2.46014853]), 'ci_99': array([1.0070823, 2.6953516])}

# ========================================================================================================================
# [4/5] comparing against: tacred_qwen_rpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 3.0586420155938363, 'main_F1': 28.99657624440347, 'other_F1': 25.937934228809635, 'bootstrap_mean_diff': 3.055636056107618, 'ci_90': array([2.30259164, 3.78750645]), 'ci_95': array([2.18361125, 3.95566009]), 'ci_99': array([1.9017498 , 4.26103432])}

# ========================================================================================================================
# [5/5] comparing against: tacred_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 4.180399773815235, 'main_F1': 28.99657624440347, 'other_F1': 24.816176470588236, 'bootstrap_mean_diff': 4.175373161775877, 'ci_90': array([3.34505581, 4.94357212]), 'ci_95': array([3.20274477, 5.12721915]), 'ci_99': array([2.96750629, 5.43036745])}
# Currently doing: 2
# MAIN_FILE:
# tacred_qwen_rpo_node_y_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - tacred_qwen_rpo_node_y_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# - tacred_qwen_rpo_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# - tacred_qwen_rpo_node_y_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# - tacred_qwen_rpo_node_y_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# - tacred_qwen_rpo_node_y_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# tacred_qwen_rpo_node_y_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 35.3 +/- 1.71	29.1 +/- 1.72	31.9 +/- 1.65
# & 35.3 & 1.71 & 29.1 & 1.72 & 31.9 & 1.65

# ========================================================================================================================
# [1/5] comparing against: tacred_qwen_rpo_node_y_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 3.620881803779369, 'main_F1': 31.923950056753693, 'other_F1': 28.303068252974324, 'bootstrap_mean_diff': 3.6447773963041983, 'ci_90': array([2.95531879, 4.39594355]), 'ci_95': array([2.80366313, 4.52251329]), 'ci_99': array([2.51408873, 4.79743896])}

# ========================================================================================================================
# [2/5] comparing against: tacred_qwen_rpo_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 2.976983833308797, 'main_F1': 31.923950056753693, 'other_F1': 28.946966223444896, 'bootstrap_mean_diff': 3.003363510179177, 'ci_90': array([2.29325528, 3.69921878]), 'ci_95': array([2.17012506, 3.83500845]), 'ci_99': array([1.85991902, 4.07627783])}

# ========================================================================================================================
# [3/5] comparing against: tacred_qwen_rpo_node_y_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 3.551480421126165, 'main_F1': 31.923950056753693, 'other_F1': 28.372469635627528, 'bootstrap_mean_diff': 3.5900135649416516, 'ci_90': array([2.85690495, 4.32290446]), 'ci_95': array([2.70200302, 4.49626622]), 'ci_99': array([2.49173706, 4.86868553])}

# ========================================================================================================================
# [4/5] comparing against: tacred_qwen_rpo_node_y_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 4.986803398955228, 'main_F1': 31.923950056753693, 'other_F1': 26.937146657798465, 'bootstrap_mean_diff': 4.99926619326563, 'ci_90': array([4.1514992 , 5.87481268]), 'ci_95': array([3.9865244 , 6.03375572]), 'ci_99': array([3.64055447, 6.36780947])}

# ========================================================================================================================
# [5/5] comparing against: tacred_qwen_rpo_node_y_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 3.0917768162277888, 'main_F1': 31.923950056753693, 'other_F1': 28.832173240525904, 'bootstrap_mean_diff': 3.117328706430622, 'ci_90': array([2.4248636 , 3.91168029]), 'ci_95': array([2.20823861, 4.03383943]), 'ci_99': array([1.97113772, 4.42311402])}
# Currently doing: 3
# MAIN_FILE:
# tacred_qwen_evoprompt_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - tacred_qwen_evoprompt_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# - tacred_qwen_evoprompt_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_qwen_evoprompt_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# - tacred_qwen_evoprompt_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# tacred_qwen_evoprompt_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 20.7 +/- 0.83	31.1 +/- 1.49	24.8 +/- 0.99
# & 20.7 & 0.83 & 31.1 & 1.49 & 24.8 & 0.99

# ========================================================================================================================
# [1/4] comparing against: tacred_qwen_evoprompt_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.239, 'alternative': 'greater', 'observed_diff': 0.17485625839161045, 'main_F1': 24.79578120153035, 'other_F1': 24.62092494313874, 'bootstrap_mean_diff': 0.17329939649113965, 'ci_90': array([-0.21772176,  0.55974971]), 'ci_95': array([-0.25643168,  0.62868708]), 'ci_99': array([-0.49111641,  0.77388246])}

# ========================================================================================================================
# [2/4] comparing against: tacred_qwen_evoprompt_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.524, 'alternative': 'greater', 'observed_diff': -0.01888910670219346, 'main_F1': 24.79578120153035, 'other_F1': 24.814670308232543, 'bootstrap_mean_diff': -0.010496214661291529, 'ci_90': array([-0.4107702 ,  0.40808247]), 'ci_95': array([-0.50006884,  0.46676794]), 'ci_99': array([-0.65015265,  0.58931643])}

# ========================================================================================================================
# [3/4] comparing against: tacred_qwen_evoprompt_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.003, 'alternative': 'greater', 'observed_diff': 0.727556312079944, 'main_F1': 24.79578120153035, 'other_F1': 24.068224889450406, 'bootstrap_mean_diff': 0.7281967523082957, 'ci_90': array([0.31654107, 1.14558463]), 'ci_95': array([0.23999295, 1.24146961]), 'ci_99': array([0.07374599, 1.43087118])}

# ========================================================================================================================
# [4/4] comparing against: tacred_qwen_evoprompt_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
# {'p_value': 0.001, 'alternative': 'greater', 'observed_diff': 0.6280869968447753, 'main_F1': 24.79578120153035, 'other_F1': 24.167694204685574, 'bootstrap_mean_diff': 0.6238815383593544, 'ci_90': array([0.24743166, 0.99456035]), 'ci_95': array([0.18125781, 1.08798126]), 'ci_99': array([0.07676137, 1.25600126])}
# Currently doing: 4
# MAIN_FILE:
# tacred_qwen_evoprompt_node_y_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - tacred_qwen_evoprompt_node_y_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_qwen_evoprompt_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_qwen_evoprompt_node_y_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# tacred_qwen_evoprompt_node_y_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 22.1 +/- 0.38	28.2 +/- 1.44	24.7 +/- 0.78
# & 22.1 & 0.38 & 28.2 & 1.44 & 24.7 & 0.78

# ========================================================================================================================
# [1/3] comparing against: tacred_qwen_evoprompt_node_y_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.151, 'alternative': 'greater', 'observed_diff': 0.19477123460015022, 'main_F1': 24.729837333636674, 'other_F1': 24.535066099036523, 'bootstrap_mean_diff': 0.19296480031856705, 'ci_90': array([-0.10033832,  0.50252685]), 'ci_95': array([-0.16587498,  0.5670419 ]), 'ci_99': array([-0.278872  ,  0.69393474])}

# ========================================================================================================================
# [2/3] comparing against: tacred_qwen_evoprompt_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.297369063130894, 'main_F1': 24.729837333636674, 'other_F1': 23.43246827050578, 'bootstrap_mean_diff': 1.2981289298845922, 'ci_90': array([0.80545783, 1.79501514]), 'ci_95': array([0.71856316, 1.87393054]), 'ci_99': array([0.56154167, 1.98109603])}

# ========================================================================================================================
# [3/3] comparing against: tacred_qwen_evoprompt_node_y_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.107, 'alternative': 'greater', 'observed_diff': 0.3179132143954817, 'main_F1': 24.729837333636674, 'other_F1': 24.411924119241192, 'bootstrap_mean_diff': 0.31701964405616306, 'ci_90': array([-0.12007123,  0.76434632]), 'ci_95': array([-0.22325898,  0.84454309]), 'ci_99': array([-0.41010679,  0.96844125])}
# Currently doing: 5
# MAIN_FILE:
# tacred_qwen_etgpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - tacred_qwen_etgpo_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_qwen_etgpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_qwen_etgpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_qwen_etgpo_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_qwen_etgpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# tacred_qwen_etgpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 21.2 +/- 0.56	43.2 +/- 1.32	28.4 +/- 0.67
# & 21.2 & 0.56 & 43.2 & 1.32 & 28.4 & 0.67

# ========================================================================================================================
# [1/5] comparing against: tacred_qwen_etgpo_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.171, 'alternative': 'greater', 'observed_diff': 0.15676839445519164, 'main_F1': 28.435130760712152, 'other_F1': 28.27836236625696, 'bootstrap_mean_diff': 0.1730010907329114, 'ci_90': array([-0.11041685,  0.46995869]), 'ci_95': array([-0.16471111,  0.53141714]), 'ci_99': array([-0.2803323 ,  0.67373207])}

# ========================================================================================================================
# [2/5] comparing against: tacred_qwen_etgpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.001, 'alternative': 'greater', 'observed_diff': 0.7315837069776521, 'main_F1': 28.435130760712152, 'other_F1': 27.7035470537345, 'bootstrap_mean_diff': 0.7417819831901047, 'ci_90': array([0.32741242, 1.16301524]), 'ci_95': array([0.27310433, 1.25794486]), 'ci_99': array([0.16838907, 1.41151925])}

# ========================================================================================================================
# [3/5] comparing against: tacred_qwen_etgpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 2.1181034794327545, 'main_F1': 28.435130760712152, 'other_F1': 26.317027281279397, 'bootstrap_mean_diff': 2.1359527953453226, 'ci_90': array([1.3903384 , 2.85671974]), 'ci_95': array([1.2589891 , 3.00516634]), 'ci_99': array([0.93562851, 3.37458122])}

# ========================================================================================================================
# [4/5] comparing against: tacred_qwen_etgpo_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.077, 'alternative': 'greater', 'observed_diff': 0.6545054403636179, 'main_F1': 28.435130760712152, 'other_F1': 27.780625320348534, 'bootstrap_mean_diff': 0.6735911218497499, 'ci_90': array([-0.11590771,  1.44514166]), 'ci_95': array([-0.28147212,  1.59245342]), 'ci_99': array([-0.50041499,  1.86680597])}

# ========================================================================================================================
# [5/5] comparing against: tacred_qwen_etgpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.251, 'alternative': 'greater', 'observed_diff': 0.13357096868442042, 'main_F1': 28.435130760712152, 'other_F1': 28.30155979202773, 'bootstrap_mean_diff': 0.14887989540245505, 'ci_90': array([-0.22758139,  0.54506378]), 'ci_95': array([-0.27454927,  0.63851163]), 'ci_99': array([-0.44657035,  0.77485547])}
# Currently doing: 6
# MAIN_FILE:
# tacred_gemma_rpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt

# COMPARE_FILES:
# - tacred_gemma_rpo_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_rpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_rpo_node_x_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# tacred_gemma_rpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 150000
# 15.1 +/- 0.63	15.0 +/- 0.49	15.0 +/- 0.54
# & 15.1 & 0.63 & 15.0 & 0.49 & 15.0 & 0.54

# ========================================================================================================================
# [1/3] comparing against: tacred_gemma_rpo_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.1650375669369861, 'main_F1': 15.04424778761062, 'other_F1': 13.879210220673635, 'bootstrap_mean_diff': 1.1749211975290281, 'ci_90': array([0.67189371, 1.72322765]), 'ci_95': array([0.56477293, 1.85229995]), 'ci_99': array([0.40585309, 1.99150016])}

# ========================================================================================================================
# [2/3] comparing against: tacred_gemma_rpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.1089886788865684, 'main_F1': 15.04424778761062, 'other_F1': 13.935259108724052, 'bootstrap_mean_diff': 1.1060665515003292, 'ci_90': array([0.51681647, 1.69093247]), 'ci_95': array([0.40484008, 1.7979752 ]), 'ci_99': array([0.25395339, 2.01747934])}

# ========================================================================================================================
# [3/3] comparing against: tacred_gemma_rpo_node_x_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.0544710859695137, 'main_F1': 15.04424778761062, 'other_F1': 13.989776701641107, 'bootstrap_mean_diff': 1.0579965452988076, 'ci_90': array([0.54789357, 1.55454515]), 'ci_95': array([0.47523314, 1.66935507]), 'ci_99': array([0.31039525, 1.88479653])}
# Currently doing: 7
# MAIN_FILE:
# tacred_gemma_rpo_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - tacred_gemma_rpo_node_y_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_rpo_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_rpo_node_y_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# tacred_gemma_rpo_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 09.8 +/- 0.27	15.6 +/- 0.79	12.0 +/- 0.43
# & 09.8 & 0.27 & 15.6 & 0.79 & 12.0 & 0.43

# ========================================================================================================================
# [1/3] comparing against: tacred_gemma_rpo_node_y_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.007, 'alternative': 'greater', 'observed_diff': 0.3635694015719899, 'main_F1': 12.023607082124638, 'other_F1': 11.660037680552648, 'bootstrap_mean_diff': 0.368715457347878, 'ci_90': array([0.12103985, 0.63245451]), 'ci_95': array([0.07149389, 0.67809085]), 'ci_99': array([-0.0291512 ,  0.76941189])}

# ========================================================================================================================
# [2/3] comparing against: tacred_gemma_rpo_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.322, 'alternative': 'greater', 'observed_diff': 0.10807163717742796, 'main_F1': 12.023607082124638, 'other_F1': 11.91553544494721, 'bootstrap_mean_diff': 0.11908471900895319, 'ci_90': array([-0.30827174,  0.53747936]), 'ci_95': array([-0.38289041,  0.60143087]), 'ci_99': array([-0.5409569 ,  0.70971691])}

# ========================================================================================================================
# [3/3] comparing against: tacred_gemma_rpo_node_y_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.1775224736637924, 'main_F1': 12.023607082124638, 'other_F1': 10.846084608460846, 'bootstrap_mean_diff': 1.184374659447224, 'ci_90': array([0.78560921, 1.57381502]), 'ci_95': array([0.73188987, 1.64960182]), 'ci_99': array([0.6284301 , 1.82070298])}
# Currently doing: 8
# MAIN_FILE:
# tacred_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - tacred_gemma_evoprompt_node_x_gradpo-prob_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_evoprompt_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_evoprompt_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# tacred_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 11.7 +/- 0.55	19.3 +/- 0.55	14.5 +/- 0.53
# & 11.7 & 0.55 & 19.3 & 0.55 & 14.5 & 0.53

# ========================================================================================================================
# [1/3] comparing against: tacred_gemma_evoprompt_node_x_gradpo-prob_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.462, 'alternative': 'greater', 'observed_diff': 0.023228566992042232, 'main_F1': 14.544745564437514, 'other_F1': 14.521516997445472, 'bootstrap_mean_diff': 0.016765919585120692, 'ci_90': array([-0.21701762,  0.24861307]), 'ci_95': array([-0.26069714,  0.29683743]), 'ci_99': array([-0.34337888,  0.3570226 ])}

# ========================================================================================================================
# [2/3] comparing against: tacred_gemma_evoprompt_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 0.6290829138351057, 'main_F1': 14.544745564437514, 'other_F1': 13.915662650602409, 'bootstrap_mean_diff': 0.6256080098920058, 'ci_90': array([0.37549216, 0.90361043]), 'ci_95': array([0.32921393, 0.95464516]), 'ci_99': array([0.25237617, 1.02062104])}

# ========================================================================================================================
# [3/3] comparing against: tacred_gemma_evoprompt_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.3992056583342265, 'main_F1': 14.544745564437514, 'other_F1': 13.145539906103288, 'bootstrap_mean_diff': 1.4030639468789328, 'ci_90': array([0.86569447, 1.98522775]), 'ci_95': array([0.73692965, 2.09544255]), 'ci_99': array([0.47476445, 2.29650134])}
# Currently doing: 9
# MAIN_FILE:
# tacred_gemma_evoprompt_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - tacred_gemma_evoprompt_node_y_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_evoprompt_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_evoprompt_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_evoprompt_node_y_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_evoprompt_node_y_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# tacred_gemma_evoprompt_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 12.6 +/- 0.59	09.0 +/- 0.49	10.5 +/- 0.39
# & 12.6 & 0.59 & 09.0 & 0.49 & 10.5 & 0.39

# ========================================================================================================================
# [1/5] comparing against: tacred_gemma_evoprompt_node_y_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.001, 'alternative': 'greater', 'observed_diff': 0.3397803986175081, 'main_F1': 10.464414957780457, 'other_F1': 10.12463455916295, 'bootstrap_mean_diff': 0.33911766643944874, 'ci_90': array([0.14044787, 0.56755874]), 'ci_95': array([0.10605511, 0.61200867]), 'ci_99': array([0.03909413, 0.67294532])}

# ========================================================================================================================
# [2/5] comparing against: tacred_gemma_evoprompt_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.107, 'alternative': 'greater', 'observed_diff': 0.2080047013702, 'main_F1': 10.464414957780457, 'other_F1': 10.256410256410257, 'bootstrap_mean_diff': 0.20445865958948406, 'ci_90': array([-0.09022355,  0.47590306]), 'ci_95': array([-0.15595495,  0.53579131]), 'ci_99': array([-0.23753894,  0.66977139])}

# ========================================================================================================================
# [3/5] comparing against: tacred_gemma_evoprompt_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 0.5434724682439516, 'main_F1': 10.464414957780457, 'other_F1': 9.920942489536506, 'bootstrap_mean_diff': 0.536149662725293, 'ci_90': array([0.28599339, 0.78777769]), 'ci_95': array([0.23908607, 0.85017303]), 'ci_99': array([0.15906626, 1.00128796])}

# ========================================================================================================================
# [4/5] comparing against: tacred_gemma_evoprompt_node_y_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.046, 'alternative': 'greater', 'observed_diff': 0.29903792849810173, 'main_F1': 10.464414957780457, 'other_F1': 10.165377029282356, 'bootstrap_mean_diff': 0.29410014981497457, 'ci_90': array([0.00322112, 0.58156231]), 'ci_95': array([-0.03937503,  0.62868404]), 'ci_99': array([-0.16345161,  0.7341691 ])}

# ========================================================================================================================
# [5/5] comparing against: tacred_gemma_evoprompt_node_y_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.44, 'alternative': 'greater', 'observed_diff': 0.04481484513613232, 'main_F1': 10.464414957780457, 'other_F1': 10.419600112644325, 'bootstrap_mean_diff': 0.03380290440217571, 'ci_90': array([-0.39693868,  0.44631847]), 'ci_95': array([-0.49764772,  0.52886041]), 'ci_99': array([-0.64554765,  0.69749636])}
# Currently doing: 10
# MAIN_FILE:
# tacred_gemma_etgpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt

# COMPARE_FILES:
# - tacred_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - tacred_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# tacred_gemma_etgpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 150000
# 11.5 +/- 0.51	18.2 +/- 0.77	14.1 +/- 0.58
# & 11.5 & 0.51 & 18.2 & 0.77 & 14.1 & 0.58

# ========================================================================================================================
# [1/2] comparing against: tacred_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.002, 'alternative': 'greater', 'observed_diff': 0.9323525203059955, 'main_F1': 14.071557065760542, 'other_F1': 13.139204545454547, 'bootstrap_mean_diff': 0.9372571418888083, 'ci_90': array([0.41009082, 1.46544489]), 'ci_95': array([0.27575931, 1.55659838]), 'ci_99': array([0.0850775 , 1.77553648])}

# ========================================================================================================================
# [2/2] comparing against: tacred_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.5799821387778419, 'main_F1': 14.071557065760542, 'other_F1': 12.4915749269827, 'bootstrap_mean_diff': 1.5732558486291641, 'ci_90': array([0.99193446, 2.20492187]), 'ci_95': array([0.88789296, 2.37535374]), 'ci_99': array([0.65340624, 2.54536406])}
# Currently doing: 11
# MAIN_FILE:
# fewrel_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - fewrel_qwen_rpo_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# - fewrel_qwen_rpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# - fewrel_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# fewrel_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 42.9 +/- 1.27	30.7 +/- 0.64	35.8 +/- 0.80
# & 42.9 & 1.27 & 30.7 & 0.64 & 35.8 & 0.80

# ========================================================================================================================
# [1/3] comparing against: fewrel_qwen_rpo_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.0134227668743208, 'main_F1': 35.833657209483086, 'other_F1': 34.820234442608765, 'bootstrap_mean_diff': 1.0102797992764485, 'ci_90': array([0.64574654, 1.39037688]), 'ci_95': array([0.59100352, 1.48198276]), 'ci_99': array([0.44198502, 1.59580073])}

# ========================================================================================================================
# [2/3] comparing against: fewrel_qwen_rpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.3557978677828757, 'main_F1': 35.833657209483086, 'other_F1': 34.47785934170021, 'bootstrap_mean_diff': 1.347989551436323, 'ci_90': array([0.94340793, 1.73343126]), 'ci_95': array([0.85695542, 1.8085521 ]), 'ci_99': array([0.74026168, 1.94762459])}

# ========================================================================================================================
# [3/3] comparing against: fewrel_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.051, 'alternative': 'greater', 'observed_diff': 0.47645802954301075, 'main_F1': 35.833657209483086, 'other_F1': 35.357199179940075, 'bootstrap_mean_diff': 0.48384545888658764, 'ci_90': array([-0.00160894,  0.94259462]), 'ci_95': array([-0.09896188,  1.05934658]), 'ci_99': array([-0.25007274,  1.29640058])}
# Currently doing: 12
# MAIN_FILE:
# fewrel_qwen_rpo_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - fewrel_qwen_rpo_node_y_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - fewrel_qwen_rpo_node_y_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - fewrel_qwen_rpo_node_y_greater-tg_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# fewrel_qwen_rpo_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 34.2 +/- 0.94	40.2 +/- 0.94	36.9 +/- 0.82
# & 34.2 & 0.94 & 40.2 & 0.94 & 36.9 & 0.82

# ========================================================================================================================
# [1/3] comparing against: fewrel_qwen_rpo_node_y_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 0.5984704558870391, 'main_F1': 36.92986758214812, 'other_F1': 36.33139712626108, 'bootstrap_mean_diff': 0.5983260353140506, 'ci_90': array([0.29812911, 0.89211382]), 'ci_95': array([0.24790169, 0.94416689]), 'ci_99': array([0.15015433, 1.05689707])}

# ========================================================================================================================
# [2/3] comparing against: fewrel_qwen_rpo_node_y_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.032, 'alternative': 'greater', 'observed_diff': 0.3402756600593406, 'main_F1': 36.92986758214812, 'other_F1': 36.58959192208878, 'bootstrap_mean_diff': 0.34965376873124027, 'ci_90': array([0.02753818, 0.6733308 ]), 'ci_95': array([-0.02106177,  0.71731812]), 'ci_99': array([-0.10416183,  0.8413087 ])}

# ========================================================================================================================
# [3/3] comparing against: fewrel_qwen_rpo_node_y_greater-tg_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.002, 'alternative': 'greater', 'observed_diff': 0.5079527011604128, 'main_F1': 36.92986758214812, 'other_F1': 36.42191488098771, 'bootstrap_mean_diff': 0.5084274592821859, 'ci_90': array([0.18827101, 0.82422839]), 'ci_95': array([0.14933116, 0.89023607]), 'ci_99': array([0.02953834, 0.99929666])}
# Currently doing: 13
# MAIN_FILE:
# fewrel_qwen_evoprompt_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - fewrel_qwen_evoprompt_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - fewrel_qwen_evoprompt_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# - fewrel_qwen_evoprompt_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - fewrel_qwen_evoprompt_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# fewrel_qwen_evoprompt_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 35.8 +/- 0.68	40.6 +/- 0.48	38.0 +/- 0.56
# & 35.8 & 0.68 & 40.6 & 0.48 & 38.0 & 0.56

# ========================================================================================================================
# [1/4] comparing against: fewrel_qwen_evoprompt_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 2.1269425094746808, 'main_F1': 38.051769413530074, 'other_F1': 35.924826904055394, 'bootstrap_mean_diff': 2.1245102314171125, 'ci_90': array([1.71532607, 2.53977525]), 'ci_95': array([1.65734297, 2.61870563]), 'ci_99': array([1.43943156, 2.7223216 ])}

# ========================================================================================================================
# [2/4] comparing against: fewrel_qwen_evoprompt_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.3537472234673587, 'main_F1': 38.051769413530074, 'other_F1': 36.698022190062716, 'bootstrap_mean_diff': 1.371892219258111, 'ci_90': array([0.95836603, 1.7985829 ]), 'ci_95': array([0.88338947, 1.87227082]), 'ci_99': array([0.72347655, 2.01850018])}

# ========================================================================================================================
# [3/4] comparing against: fewrel_qwen_evoprompt_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 2.1373493300697888, 'main_F1': 38.051769413530074, 'other_F1': 35.914420083460286, 'bootstrap_mean_diff': 2.135728092144564, 'ci_90': array([1.72818145, 2.55274617]), 'ci_95': array([1.6178269 , 2.63511719]), 'ci_99': array([1.45481673, 2.75474929])}

# ========================================================================================================================
# [4/4] comparing against: fewrel_qwen_evoprompt_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.8294455940538725, 'main_F1': 38.051769413530074, 'other_F1': 36.2223238194762, 'bootstrap_mean_diff': 1.83163461941645, 'ci_90': array([1.40050362, 2.25448454]), 'ci_95': array([1.34595256, 2.33960638]), 'ci_99': array([1.18312055, 2.44434418])}
# Currently doing: 14
# MAIN_FILE:
# fewrel_qwen_etgpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt

# COMPARE_FILES:
# - fewrel_qwen_etgpo_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# - fewrel_qwen_etgpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# - fewrel_qwen_etgpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# fewrel_qwen_etgpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 150000
# 36.2 +/- 1.44	35.1 +/- 1.21	35.7 +/- 1.24
# & 36.2 & 1.44 & 35.1 & 1.21 & 35.7 & 1.24

# ========================================================================================================================
# [1/3] comparing against: fewrel_qwen_etgpo_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 0.6295464473765122, 'main_F1': 35.67801773745853, 'other_F1': 35.04847129008202, 'bootstrap_mean_diff': 0.6326300119742927, 'ci_90': array([0.36182325, 0.90161841]), 'ci_95': array([0.3148561 , 0.95462477]), 'ci_99': array([0.21247737, 1.10526501])}

# ========================================================================================================================
# [2/3] comparing against: fewrel_qwen_etgpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.004, 'alternative': 'greater', 'observed_diff': 0.5562570915410916, 'main_F1': 35.67801773745853, 'other_F1': 35.12176064591744, 'bootstrap_mean_diff': 0.5734876754484151, 'ci_90': array([0.20801977, 0.93691615]), 'ci_95': array([0.14820594, 1.01529757]), 'ci_99': array([0.04471441, 1.11600001])}

# ========================================================================================================================
# [3/3] comparing against: fewrel_qwen_etgpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.02, 'alternative': 'greater', 'observed_diff': 0.43117502792925677, 'main_F1': 35.67801773745853, 'other_F1': 35.246842709529275, 'bootstrap_mean_diff': 0.448151180542694, 'ci_90': array([0.08826586, 0.83112346]), 'ci_95': array([0.01266947, 0.91231182]), 'ci_99': array([-0.11851814,  0.9893099 ])}
# Currently doing: 15
# MAIN_FILE:
# fewrel_gemma_rpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt

# COMPARE_FILES:
# - fewrel_gemma_rpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# - fewrel_gemma_rpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - fewrel_gemma_rpo_node_x_greater_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# fewrel_gemma_rpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 150000
# 32.8 +/- 1.11	43.8 +/- 2.03	37.5 +/- 1.30
# & 32.8 & 1.11 & 43.8 & 2.03 & 37.5 & 1.30

# ========================================================================================================================
# [1/3] comparing against: fewrel_gemma_rpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 7.165769204742716, 'main_F1': 37.48213366874393, 'other_F1': 30.31636446400121, 'bootstrap_mean_diff': 7.139283742704858, 'ci_90': array([6.29715044, 7.97616654]), 'ci_95': array([6.12886531, 8.07014268]), 'ci_99': array([5.86209445, 8.28524969])}

# ========================================================================================================================
# [2/3] comparing against: fewrel_gemma_rpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 4.772882567422336, 'main_F1': 37.48213366874393, 'other_F1': 32.70925110132159, 'bootstrap_mean_diff': 4.752673347115404, 'ci_90': array([4.00829172, 5.48851401]), 'ci_95': array([3.87707362, 5.61382418]), 'ci_99': array([3.58440437, 5.92464714])}

# ========================================================================================================================
# [3/3] comparing against: fewrel_gemma_rpo_node_x_greater_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 7.333287982231173, 'main_F1': 37.48213366874393, 'other_F1': 30.148845686512754, 'bootstrap_mean_diff': 7.308256652659127, 'ci_90': array([6.43828065, 8.13478646]), 'ci_95': array([6.31489453, 8.26244534]), 'ci_99': array([5.95564716, 8.47452029])}
# Currently doing: 16
# MAIN_FILE:
# fewrel_gemma_rpo_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt

# COMPARE_FILES:
# - fewrel_gemma_rpo_node_y_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - fewrel_gemma_rpo_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - fewrel_gemma_rpo_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# fewrel_gemma_rpo_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# 150000
# 32.5 +/- 1.28	34.1 +/- 1.77	33.3 +/- 1.40
# & 32.5 & 1.28 & 34.1 & 1.77 & 33.3 & 1.40

# ========================================================================================================================
# [1/3] comparing against: fewrel_gemma_rpo_node_y_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.0929483228096188, 'main_F1': 33.27041978522616, 'other_F1': 32.17747146241654, 'bootstrap_mean_diff': 1.0905827868432363, 'ci_90': array([0.66346972, 1.51326652]), 'ci_95': array([0.56664172, 1.58286193]), 'ci_99': array([0.46413743, 1.72919351])}

# ========================================================================================================================
# [2/3] comparing against: fewrel_gemma_rpo_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.4459225274565028, 'main_F1': 33.27041978522616, 'other_F1': 31.824497257769657, 'bootstrap_mean_diff': 1.4484078670158822, 'ci_90': array([0.99372115, 1.92786522]), 'ci_95': array([0.9347492 , 1.98821535]), 'ci_99': array([0.743879  , 2.16040374])}

# ========================================================================================================================
# [3/3] comparing against: fewrel_gemma_rpo_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.018, 'alternative': 'greater', 'observed_diff': 0.5388049787360245, 'main_F1': 33.27041978522616, 'other_F1': 32.731614806490136, 'bootstrap_mean_diff': 0.5431197327006182, 'ci_90': array([0.13280417, 0.95552652]), 'ci_95': array([0.04208542, 1.03328982]), 'ci_99': array([-0.09628624,  1.14034056])}
# Currently doing: 17
# MAIN_FILE:
# fewrel_gemma_evoprompt_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt

# COMPARE_FILES:
# - fewrel_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - fewrel_gemma_evoprompt_node_x_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# - fewrel_gemma_evoprompt_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - fewrel_gemma_evoprompt_node_x_greater_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# - fewrel_gemma_evoprompt_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# fewrel_gemma_evoprompt_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 150000
# 37.3 +/- 1.43	30.7 +/- 1.91	33.7 +/- 1.71
# & 37.3 & 1.43 & 30.7 & 1.91 & 33.7 & 1.71

# ========================================================================================================================
# [1/5] comparing against: fewrel_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 5.765395568183113, 'main_F1': 33.65814345373287, 'other_F1': 27.892747885549756, 'bootstrap_mean_diff': 5.778781936896141, 'ci_90': array([5.14829676, 6.40404357]), 'ci_95': array([4.99353599, 6.54743592]), 'ci_99': array([4.78052778, 6.82384569])}

# ========================================================================================================================
# [2/5] comparing against: fewrel_gemma_evoprompt_node_x_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 1.7109062623872582, 'main_F1': 33.65814345373287, 'other_F1': 31.94723719134561, 'bootstrap_mean_diff': 1.7112220845736241, 'ci_90': array([1.21784095, 2.22009468]), 'ci_95': array([1.13942667, 2.30953925]), 'ci_99': array([0.98280374, 2.47542755])}

# ========================================================================================================================
# [3/5] comparing against: fewrel_gemma_evoprompt_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 5.35323940256016, 'main_F1': 33.65814345373287, 'other_F1': 28.30490405117271, 'bootstrap_mean_diff': 5.362343864968044, 'ci_90': array([4.73853197, 5.96294978]), 'ci_95': array([4.60225169, 6.08212162]), 'ci_99': array([4.40356963, 6.35445552])}

# ========================================================================================================================
# [4/5] comparing against: fewrel_gemma_evoprompt_node_x_greater_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 0.9098755553494939, 'main_F1': 33.65814345373287, 'other_F1': 32.748267898383375, 'bootstrap_mean_diff': 0.9202311434545827, 'ci_90': array([0.45899603, 1.35440728]), 'ci_95': array([0.38891358, 1.4196436 ]), 'ci_99': array([0.21114603, 1.66374426])}

# ========================================================================================================================
# [5/5] comparing against: fewrel_gemma_evoprompt_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 2.7924574156330415, 'main_F1': 33.65814345373287, 'other_F1': 30.865686038099827, 'bootstrap_mean_diff': 2.8008146872001665, 'ci_90': array([2.23996739, 3.36189254]), 'ci_95': array([2.13671018, 3.48734698]), 'ci_99': array([1.93891523, 3.71832718])}
# Currently doing: 18
# MAIN_FILE:
# fewrel_gemma_evoprompt_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt

# COMPARE_FILES:
# - fewrel_gemma_evoprompt_node_y_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# - fewrel_gemma_evoprompt_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# - fewrel_gemma_evoprompt_node_y_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# fewrel_gemma_evoprompt_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 150000
# 35.7 +/- 1.09	39.5 +/- 1.73	37.5 +/- 1.33
# & 35.7 & 1.09 & 39.5 & 1.73 & 37.5 & 1.33

# ========================================================================================================================
# [1/3] comparing against: fewrel_gemma_evoprompt_node_y_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 2.330489986878206, 'main_F1': 37.49207556738937, 'other_F1': 35.161585580511165, 'bootstrap_mean_diff': 2.3444919398430266, 'ci_90': array([1.83813084, 2.88715255]), 'ci_95': array([1.70126427, 2.98523343]), 'ci_99': array([1.50748761, 3.08870492])}

# ========================================================================================================================
# [2/3] comparing against: fewrel_gemma_evoprompt_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 2.334225436730428, 'main_F1': 37.49207556738937, 'other_F1': 35.15785013065894, 'bootstrap_mean_diff': 2.3493744437992943, 'ci_90': array([1.83969887, 2.87710514]), 'ci_95': array([1.75855821, 2.95982058]), 'ci_99': array([1.57703783, 3.10414454])}

# ========================================================================================================================
# [3/3] comparing against: fewrel_gemma_evoprompt_node_y_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.008, 'alternative': 'greater', 'observed_diff': 0.7481304825557942, 'main_F1': 37.49207556738937, 'other_F1': 36.74394508483358, 'bootstrap_mean_diff': 0.7488891454983456, 'ci_90': array([0.24376963, 1.27177478]), 'ci_95': array([0.12836151, 1.38335311]), 'ci_99': array([-0.13905904,  1.53957155])}
# Currently doing: 19
# MAIN_FILE:
# fewrel_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt

# COMPARE_FILES:
# - fewrel_gemma_etgpo_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# - fewrel_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# fewrel_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# 150000
# 33.8 +/- 1.50	36.6 +/- 2.14	35.2 +/- 1.73
# & 33.8 & 1.50 & 36.6 & 2.14 & 35.2 & 1.73

# ========================================================================================================================
# [1/2] comparing against: fewrel_gemma_etgpo_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.132, 'alternative': 'greater', 'observed_diff': 0.19939977871560188, 'main_F1': 35.16596180956042, 'other_F1': 34.96656203084482, 'bootstrap_mean_diff': 0.19737549797006806, 'ci_90': array([-0.10878917,  0.50685076]), 'ci_95': array([-0.15111837,  0.5652553 ]), 'ci_99': array([-0.27865399,  0.67546792])}

# ========================================================================================================================
# [2/2] comparing against: fewrel_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
# {'p_value': 0.0, 'alternative': 'greater', 'observed_diff': 7.348255177740754, 'main_F1': 35.16596180956042, 'other_F1': 27.817706631819668, 'bootstrap_mean_diff': 7.3198779106726315, 'ci_90': array([6.66833435, 8.00879467]), 'ci_95': array([6.59376427, 8.09976561]), 'ci_99': array([6.32786932, 8.33635328])}

In [ ]:
$^{\dagger}$ 
# summary_rows = []
# for filename, res in results.items():
#     summary_rows.append({
#         'file': filename,
#         'main_F1': res['main_F1'],
#         'other_F1': res['other_F1'],
#         'diff_main_minus_other': res['observed_diff'],
#         'p_value': res['p_value'],
#         'alternative': res['alternative'],
#         'ci95_low': float(res['ci_95'][0]),
#         'ci95_high': float(res['ci_95'][1]),
#     })

# try:
#     import pandas as pd
#     summary = pd.DataFrame(summary_rows).sort_values('p_value')
#     display(summary)
# except ImportError:
#     summary = sorted(summary_rows, key=lambda x: x['p_value'])
#     for row in summary:
#         print(row)
